# KADaP GPUaaS 리포지토리 실습 — 퀵스타트

이 노트북은 저장소의 `src/` 스크립트들을 순서대로 실행하며 결과를 확인합니다.
스크립트를 직접 터미널에서 돌려도 되지만, 노트북으로 실행하면 출력이 파일에 함께 저장되어
실습 기록을 남기기 좋습니다.

**실행 전 확인**: 이 노트북이 저장소 루트 기준 `notebooks/` 안에 있는지 확인하세요.

## 0. 저장소 위치 확인

In [ ]:
import os, sys, subprocess

# 노트북의 현재 위치에서 저장소 루트 찾기
here = os.getcwd()
root = here if os.path.isdir(os.path.join(here, "src")) else os.path.dirname(here)
print("현재 위치   :", here)
print("저장소 루트 :", root)
print("src 존재    :", os.path.isdir(os.path.join(root, "src")))
print("data 존재   :", os.path.isdir(os.path.join(root, "data")))

SRC = os.path.join(root, "src")

def run(script, env=None, args=None):
    """스크립트를 실행하고 출력을 그대로 표시."""
    e = dict(os.environ)
    if env: e.update({k: str(v) for k, v in env.items()})
    cmd = [sys.executable, os.path.join(SRC, script)] + (args or [])
    r = subprocess.run(cmd, capture_output=True, text=True, env=e, cwd=root)
    print(r.stdout)
    if r.returncode != 0:
        print("--- STDERR ---"); print(r.stderr[-1500:])
    return r.returncode

## 1. 환경 확인 (실습 00)

할당된 GPU와 VRAM, 마이디스크 마운트 경로를 확인합니다.

In [ ]:
run("00_env_report.py")

## 2. 마운트 경로 관찰 (실습 01) — 핵심

워크로드 생성 시 연결한 소스코드·데이터셋 경로·모델 경로가 컨테이너 안에서
각각 어디에 붙었는지 확인합니다. 이 저장소가 마운트된 위치가
'소스코드 추가' 화면에서 입력한 마운트 경로와 일치하는지 대조해보세요.

In [ ]:
run("01_mount_probe.py")

## 3. 데이터셋 로드 (실습 02)

In [ ]:
run("02_dataset_read.py")

## 4. 학습 및 체크포인트 저장 (실습 03)

In [ ]:
run("03_train_checkpoint.py", env={"KADAP_EPOCHS": 300})

## 5. 체크포인트 복원 및 예측 (실습 04)

저장한 모델을 다시 불러와 예측이 재현되는지 확인합니다.

In [ ]:
run("04_resume_from_model.py")

## 6. 설정 주입 확인 (실습 05)

환경변수와 명령행 인자의 우선순위를 확인합니다.
아래는 환경변수로 epochs를, 명령행 인자로 lr을 지정한 경우입니다.

In [ ]:
run("05_params_env.py", env={"KADAP_EPOCHS": 500, "KADAP_TAG": "exp-001"}, args=["--lr", "0.01"])

## 7. 저장된 리포트 확인

각 스크립트가 남긴 JSON 리포트 목록입니다.

In [ ]:
import json, glob
sys.path.insert(0, SRC)
import kadap_util as k

base = k.find_mydisk() or "/tmp"
rep_dir = os.path.join(base, "practice", "reports")
print("리포트 폴더:", rep_dir)
print()
files = sorted(glob.glob(os.path.join(rep_dir, "*.json")))
for f in files:
    print(f"  {os.path.basename(f):<44} {os.path.getsize(f):>8,} bytes")
print()
print(f"총 {len(files)}개")

## 체크리스트

- [ ] 00: 할당된 GPU 종류와 VRAM을 확인했다
- [ ] 01: 이 저장소가 마운트된 경로를 확인했다 (소스코드 추가 시 입력값과 대조)
- [ ] 01: 데이터셋/모델 경로를 별도 연결했다면 그 마운트 지점도 확인했다
- [ ] 02: 데이터 300건이 정상 로드됐다
- [ ] 03: 체크포인트가 저장됐고, 저장 경로가 마이디스크인지 확인했다
- [ ] 04: 복원한 모델의 예측이 정상 동작했다
- [ ] 05: 환경변수 주입이 반영되는 것을 확인했다
- [ ] 워크로드 종료 전 Ctrl+S 로 이 노트북을 저장했다

## 다음 단계

1. 데이터셋 경로·모델 경로를 실제로 등록해 연결한 뒤 01·03을 재실행 → 마운트 지점 변화 관찰
2. 워크스페이스 > 공유 리포지토리에 등록해 팀 공유 동작 확인
3. '종료 시 이미지로 저장'으로 프라이빗 레지스트리에 컨테이너 이미지 생성
4. Batch Job 배정 시 파라미터 방식으로 05 재실행